In [10]:
import argparse
import json
from pathlib import Path
import torch
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, BertForTokenClassification

In [11]:
def parse_args(args=None):
    parser = argparse.ArgumentParser(description="Generate PKU-SEGPOS span_list predictions.")
    parser.add_argument("--input", default="test_public.jsonline")
    parser.add_argument("--output", default="PKUSEGPOS.jsonline")
    parser.add_argument("--train", default="train.jsonline")
    parser.add_argument("--model", default="model")
    parser.add_argument("--batch-size", type=int, default=32)
    parser.add_argument("--max-len", type=int, default=128)
    args, _ = parser.parse_known_args(args)
    return args

In [12]:
def build_label(train_data):
    uniquetypes = set()
    for item in train_data:
        for span in item['span_list']:
            uniquetypes.add(span['type'])
    sortedtypes = sorted(list(uniquetypes))
    label_to_id = {"O": 0}
    for t in sortedtypes:
        label_to_id[f"B-{t}"] = len(label_to_id)
        label_to_id[f"I-{t}"] = len(label_to_id)
    id_to_label = {values:keys for keys,values in label_to_id.items()}
    return label_to_id, id_to_label

In [13]:
def load_jsonline(path):
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                data.append(json.loads(line))
    return data

In [14]:
class Tokendataset(Dataset):
    def __init__(self, data, tokenizer, max_len):
        self.data = data
        self.tokenizer = tokenizer
        self.max_len = max_len
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        tokens = self.data[idx]["tokens"]
        encoding = self.tokenizer(
            tokens,
            is_split_into_words=True,
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt",
        )
        return {
            "idx": idx,
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
        }

In [15]:
def bio2spans(labels):
    spans = []
    start = None
    current_type = None
    def closespan(end):
        if start is not None:
            spans.append({"start": start, "end": end, "type": current_type})
    for idx, label in enumerate(labels):
        if label == "O" or "-" not in label:
            closespan(idx - 1)
            start = None
            current_type = None
            continue
        prefix, tag_type = label.split("-", 1)
        if prefix == "B" or current_type != tag_type:
            closespan(idx - 1)
            start = idx
            current_type = tag_type
        elif prefix == "I" and start is None:
            start = idx
            current_type = tag_type
    closespan(len(labels) - 1)
    return spans

In [16]:
def predict(data, model, tokenizer, id_to_label, device, batch_size, max_len):
    dataset = Tokendataset(data, tokenizer, max_len)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    predictions = [None] * len(data)
    model.eval()
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            pred_ids = torch.argmax(outputs.logits, dim=-1).cpu()
            for row, item_idx in enumerate(batch["idx"].tolist()):
                word_ids = tokenizer(
                    data[item_idx]["tokens"],
                    is_split_into_words=True,
                    padding="max_length",
                    truncation=True,
                    max_length=max_len,
                ).word_ids()
                labels = []
                seen_word_ids = set()
                for token_pos, word_idx in enumerate(word_ids):
                    if word_idx is None or word_idx in seen_word_ids:
                        continue
                    seen_word_ids.add(word_idx)
                    labels.append(id_to_label[pred_ids[row][token_pos].item()])
                predictions[item_idx] = labels
    return predictions

In [17]:
def write_predictions(data, labels_by_item, output_path):
    with open(output_path, "w", encoding="utf-8") as f:
        for item, labels in zip(data, labels_by_item):
            output_item = dict(item)
            output_item["span_list"] = bio2spans(labels)
            f.write(json.dumps(output_item, ensure_ascii=False) + "\n")

In [18]:
args = parse_args()
train_data = load_jsonline(args.train)
input_data = load_jsonline(args.input)
_, id_to_label = build_label(train_data)

model_path = Path(args.model)
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = BertForTokenClassification.from_pretrained(model_path)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
labels_by_item = predict(
    input_data,
    model,
    tokenizer,
    id_to_label,
    device,
    batch_size=args.batch_size,
    max_len=args.max_len,
)
write_predictions(input_data, labels_by_item, args.output)